# 02 — CheXpert v1: MedGemma Inference
Zero-shot PTX classification + report generation.
Resume-safe: existing results in JSONL are skipped automatically.

In [1]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src2.utils import load_config, load_done_ids
from src2.data import load_chexpert_v1_benchmark
from src2.medgemma_runner import MedGemmaRunner

CFG         = load_config(PROJECT_ROOT / 'configs2' / 'server.yaml')
DATA_ROOT   = Path(CFG['paths']['data_root'])
OUTPUT_ROOT = Path(CFG['paths']['output_root'])
OUT_JSONL   = OUTPUT_ROOT / CFG['outputs']['chexpert_v1']['medgemma']
OUT_JSONL.parent.mkdir(parents=True, exist_ok=True)

print('DATA_ROOT  :', DATA_ROOT)
print('Output JSONL:', OUT_JSONL)

/data/arta/envs/thoraxai/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DATA_ROOT  : /data/arta/thoraxai_research/1_equity/data3
Output JSONL: /data/arta/thoraxai_research/2_srg/hasil_penelitian_2/outputs/chexpert_v1/medgemma.jsonl


In [2]:
df = load_chexpert_v1_benchmark(DATA_ROOT, seed=42)

done_ids = load_done_ids(OUT_JSONL)
print(f'Total: {len(df)} | Already done: {len(done_ids)} | Remaining: {len(df) - len(done_ids)}')

Total: 1050 | Already done: 0 | Remaining: 1050


In [3]:
runner = MedGemmaRunner(
    model_path=CFG['models']['medgemma'],
    max_new_tokens=CFG['inference']['max_new_tokens'],
)

[MedGemma] Loading: google/medgemma-1.5-4b-it
[MedGemma] dtype: bfloat16 (mandatory for Gemma family)


The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100%|██████████| 883/883 [00:00<00:00, 1116.32it/s]


[MedGemma] Ready. VRAM: 8.60 GB


In [4]:
runner.run_dataset(df, OUT_JSONL, dataset_name='chexpert_v1', done_ids=done_ids)

  [1/1050] patient22703_study1_view1_frontal ... 

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


✓ diag=1 conf=8 (1.51s)
  [2/1050] patient29495_study4_view1_frontal ... 

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


✓ diag=0 conf=9 (1.46s)
  [3/1050] patient39640_study1_view1_frontal ... 

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


... (2,091 outputs left out)

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


✓ diag=0 conf=9 (1.59s)
  [1050/1050] patient48779_study1_view1_frontal ... 

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


✓ diag=0 conf=9 (1.39s)

[MedGemma] Done. Skipped 0/1050 (already done).


In [5]:
runner.free()
print('Done. JSONL:', OUT_JSONL)

Done. JSONL: /data/arta/thoraxai_research/2_srg/hasil_penelitian_2/outputs/chexpert_v1/medgemma.jsonl
